# M5 Demand Forecasting — Models and Evaluation

MGMT 599 Group 5 · Walmart M5 dataset

**What this notebook does**
1. Pulls the scoped subset from Athena (or local Parquet as a fallback)
2. Holds out the final 28 days — nothing after the cutoff is used for training
3. Fits four models: naive, seasonal naive, weekday mean, gradient boosting
4. Scores all four on the same holdout with RMSE, MAE and bias
5. Writes predictions back to S3 so QuickSight can chart forecast vs actual

**Scope decision:** the pipeline runs on all 10 stores, but models are trained on
one category (FOODS) in one state to keep iteration fast. Full-population
modelling is not what the rubric rewards and would consume the whole week.

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

# ---- config ----
BUCKET      = "mgmt599-group5-m5"
DATABASE    = "m5_db"
REGION      = "us-east-1"
HORIZON     = 28          # forecast window, matches the M5 competition
SCOPE_CAT   = "FOODS"     # the category we model
SCOPE_STATE = "CA"        # partition filter keeps the Athena scan small

## 2. Load the data

Two paths. Use Athena if `awswrangler` is installed and credentials are configured
— that is the version to screenshot for the report, since it proves the pipeline
is being queried end to end. Fall back to local Parquet if this is running
somewhere without AWS access.

In [ ]:
QUERY = f"""
SELECT item_id, store_id, state_id, cat_id, dept_id,
       date, day_num, wday, weekday, month, year,
       units_sold, sell_price, snap_flag, has_event
FROM {DATABASE}.sales_long
WHERE state_id = '{SCOPE_STATE}'
  AND cat_id   = '{SCOPE_CAT}'
"""

try:
    import awswrangler as wr
    df = wr.athena.read_sql_query(
        QUERY,
        database=DATABASE,
        s3_output=f"s3://{BUCKET}/athena-results/",
    )
    print("loaded from Athena")
except Exception as e:
    print(f"Athena unavailable ({type(e).__name__}), falling back to local Parquet")
    df = pd.read_parquet("../data/processed/")
    df = df[(df.state_id == SCOPE_STATE) & (df.cat_id == SCOPE_CAT)]

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["item_id", "store_id", "date"]).reset_index(drop=True)

print(f"rows      : {len(df):,}")
print(f"items     : {df.item_id.nunique():,}")
print(f"stores    : {df.store_id.nunique()}")
print(f"date range: {df.date.min().date()} to {df.date.max().date()}")
df.head()

## 3. Exploratory checks

These charts go straight into the checkpoint as notebook-output evidence.

In [ ]:
daily = df.groupby("date")["units_sold"].sum()

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(daily.index, daily.values, lw=0.7, color="#2E74B5", label="daily total")
ax.plot(daily.index, daily.rolling(28).mean(), lw=1.8, color="#D98A4E",
        label="28-day moving average")
ax.set_title(f"Daily units sold — {SCOPE_CAT}, {SCOPE_STATE}")
ax.set_ylabel("units")
ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))

# weekday seasonality — usually the strongest single pattern in this data
order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
wk = df.groupby("weekday")["units_sold"].mean()
wk = wk.reindex([d for d in order if d in wk.index])
axes[0].bar(range(len(wk)), wk.values, color="#5b8db8")
axes[0].set_xticks(range(len(wk)))
axes[0].set_xticklabels([d[:3] for d in wk.index])
axes[0].set_title("Avg units by weekday")

# intermittency — the defining characteristic of M5
zero_share = (df.units_sold == 0).mean() * 100
counts = df.units_sold.clip(upper=10).value_counts().sort_index()
axes[1].bar(counts.index, counts.values, color="#7aa87a")
axes[1].set_title(f"Units per item-day\n{zero_share:.1f}% are zero")
axes[1].set_xlabel("units (capped at 10)")

# SNAP effect
snap = df.groupby("snap_flag")["units_sold"].mean()
axes[2].bar(["Normal day", "SNAP day"], snap.values, color=["#b8bcc2", "#D98A4E"])
lift = 100 * (snap.get(1, 0) - snap.get(0, 0)) / snap.get(0, 1)
axes[2].set_title(f"SNAP day effect ({lift:+.1f}%)")

plt.tight_layout(); plt.show()
print(f"Zero-sales item-days: {zero_share:.1f}%")
print(f"SNAP lift: {lift:+.1f}%")

## 4. Train / test split

The final 28 days are held out. Nothing on or after the cutoff is visible to any
model during fitting — this is what makes the comparison honest.

In [ ]:
cutoff = df["date"].max() - pd.Timedelta(days=HORIZON - 1)
train = df[df["date"] <  cutoff].copy()
test  = df[df["date"] >= cutoff].copy()

print(f"cutoff : {cutoff.date()}")
print(f"train  : {len(train):,} rows, through {train.date.max().date()}")
print(f"test   : {len(test):,} rows, {test.date.min().date()} to {test.date.max().date()}")
assert test.date.nunique() == HORIZON, "holdout is not exactly HORIZON days"

## 5. Metrics

Reporting **bias** alongside RMSE matters here. For an inventory manager the cost
of a stockout is not symmetric with the cost of excess stock, so a model that is
accurate on average but systematically under-forecasts is worse than its RMSE
alone suggests. That asymmetry is worth a paragraph in the limitations section.

In [ ]:
def rmse(a, f): return float(np.sqrt(np.mean((np.asarray(a, float) - np.asarray(f, float))**2)))
def mae(a, f):  return float(np.mean(np.abs(np.asarray(a, float) - np.asarray(f, float))))
def bias(a, f): return float(np.mean(np.asarray(f, float) - np.asarray(a, float)))

def score(name, actual, pred):
    return {"model": name, "rmse": rmse(actual, pred),
            "mae": mae(actual, pred), "bias": bias(actual, pred)}

series_key = ["item_id", "store_id"]
scores, predictions = [], {}

### Model 1 — Naive (carry the last observed value forward)

In [ ]:
last_val = train.groupby(series_key)["units_sold"].last()
pred = test.set_index(series_key).index.map(last_val).to_numpy()
pred = np.nan_to_num(pred.astype(float), nan=0.0)

predictions["naive_last_value"] = pred
scores.append(score("naive_last_value", test["units_sold"], pred))
scores[-1]

### Model 2 — Seasonal naive (same weekday, 4 weeks earlier)

The benchmark to beat. Ten minutes to write and genuinely hard to outperform on
intermittent retail demand.

In [ ]:
hist = df.set_index(["item_id", "store_id", "date"])["units_sold"].to_dict()
pred = np.array([
    hist.get((i, s, d - pd.Timedelta(days=28)), 0.0)
    for i, s, d in zip(test["item_id"], test["store_id"], test["date"])
], dtype=float)

predictions["seasonal_naive_28d"] = pred
scores.append(score("seasonal_naive_28d", test["units_sold"], pred))
scores[-1]

### Model 3 — Per-item weekday mean from the training period

In [ ]:
wday_mean = train.groupby(["item_id", "store_id", "wday"])["units_sold"].mean()
pred = np.array([
    wday_mean.get((i, s, w), 0.0)
    for i, s, w in zip(test["item_id"], test["store_id"], test["wday"])
], dtype=float)

predictions["weekday_mean"] = pred
scores.append(score("weekday_mean", test["units_sold"], pred))
scores[-1]

### Model 4 — Gradient boosting on calendar, price and lag features

Lags are computed on the full frame then split. Using lags of 28 days or more
means no test row can see an actual value from inside the holdout window, so
there is no leakage.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

FEATURES = ["wday", "month", "snap_flag", "has_event", "sell_price",
            "lag_28", "lag_35", "roll_28"]

full = df.sort_values(series_key + ["date"]).copy()
g = full.groupby(series_key)["units_sold"]
full["lag_28"]  = g.shift(28)
full["lag_35"]  = g.shift(35)
# transform keeps the original index — safer than reset_index gymnastics
full["roll_28"] = g.transform(lambda s: s.shift(28).rolling(28, min_periods=1).mean())

tr = full[full["date"] <  cutoff].dropna(subset=["lag_28", "lag_35"])
te = full[full["date"] >= cutoff].copy()
te[["lag_28", "lag_35", "roll_28"]] = te[["lag_28", "lag_35", "roll_28"]].fillna(0)

model = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.08, random_state=42)
model.fit(tr[FEATURES], tr["units_sold"])

te = te.sort_values(series_key + ["date"])
te["forecast"] = np.clip(model.predict(te[FEATURES]), 0, None)

predictions["gradient_boosting"] = te["forecast"].to_numpy()
scores.append(score("gradient_boosting", te["units_sold"], te["forecast"]))
print(f"trained on {len(tr):,} rows")
scores[-1]

## 6. Scorecard

If the seasonal naive baseline wins, report that honestly. It is a legitimate and
well-documented result for intermittent demand, not a project failure — and
saying so is stronger than quietly dropping the baseline from the report.

In [ ]:
scorecard = pd.DataFrame(scores).sort_values("rmse").reset_index(drop=True)
for c in ["rmse", "mae", "bias"]:
    scorecard[c] = scorecard[c].round(4)

best = scorecard.iloc[0]["model"]
print(f"Best model by RMSE: {best}\n")
scorecard

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
colors = ["#D98A4E" if m == best else "#5b8db8" for m in scorecard["model"]]
ax.barh(scorecard["model"], scorecard["rmse"], color=colors)
ax.invert_yaxis()
ax.set_xlabel("RMSE on the 28-day holdout (lower is better)")
ax.set_title("Model comparison")
for i, v in enumerate(scorecard["rmse"]):
    ax.text(v, i, f" {v:.3f}", va="center", fontsize=9)
plt.tight_layout(); plt.show()

## 7. Forecast vs actual over the holdout window

In [ ]:
actual_daily = test.groupby("date")["units_sold"].sum()

fig, ax = plt.subplots(figsize=(12, 4.2))
ax.plot(actual_daily.index, actual_daily.values, "o-", lw=2,
        color="#2d3436", label="Actual", zorder=5)

for name, pred in predictions.items():
    frame = te if name == "gradient_boosting" else test
    tmp = pd.DataFrame({"date": frame["date"].to_numpy(),
                        "f": np.asarray(pred, float)})
    daily_f = tmp.groupby("date")["f"].sum()
    ax.plot(daily_f.index, daily_f.values, lw=1.4, alpha=0.85, label=name)

ax.set_title(f"Forecast vs actual — {SCOPE_CAT}, {SCOPE_STATE}, {HORIZON}-day holdout")
ax.set_ylabel("total units/day")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 8. Write predictions back to S3

**Do not skip this.** QuickSight can only chart what lives in Athena. If the
forecasts exist only as a DataFrame in this notebook, the forecast-vs-actual view
promised in the proposal cannot be built.

After this runs, execute `sql/08_forecast_vs_actual.sql` to register the table.

In [ ]:
frames = []
for name, pred in predictions.items():
    frame = te if name == "gradient_boosting" else test
    frames.append(pd.DataFrame({
        "date":           pd.to_datetime(frame["date"].to_numpy()),
        "store_id":       frame["store_id"].to_numpy(),
        "cat_id":         frame["cat_id"].to_numpy(),
        "item_id":        frame["item_id"].to_numpy(),
        "actual_units":   frame["units_sold"].to_numpy().astype(float),
        "forecast_units": np.asarray(pred, float),
        "model_name":     name,
    }))

out = pd.concat(frames, ignore_index=True)
print(f"{len(out):,} prediction rows across {out.model_name.nunique()} models")

# local copy first, so we always have the artifact even if the upload fails
out.to_parquet("forecast_vs_actual.parquet", index=False, compression="snappy")
scorecard.to_csv("model_scorecard.csv", index=False)

try:
    import awswrangler as wr
    wr.s3.to_parquet(
        df=out,
        path=f"s3://{BUCKET}/curated/forecast_vs_actual/",
        dataset=True,
        mode="overwrite",
        compression="snappy",
    )
    print("uploaded to S3")
except Exception as e:
    print(f"S3 upload skipped ({type(e).__name__}). Upload manually:")
    print(f"  aws s3 cp forecast_vs_actual.parquet s3://{BUCKET}/curated/forecast_vs_actual/")

out.head()

## 9. Findings to carry into the report

Fill these in from the numbers above rather than from memory:

1. **Demand is highly intermittent** — X% of item-days have zero sales, which is
   what makes point forecasting at item level hard and pushes error metrics around.
2. **Day of week is the dominant pattern**, which is why the seasonal naive
   baseline is competitive.
3. **SNAP days lift FOODS demand by X%**, and the effect is much weaker for other
   categories — evidence the signal is real rather than an artifact.
4. **Best model: [name], RMSE X.XX vs baseline X.XX** — a Y% improvement (or: the
   baseline was not beaten, which is consistent with the intermittent-demand
   literature).
5. **Bias direction matters** — the best model under/over-forecasts by X units per
   item-day on average, which for an inventory manager translates to
   [stockout risk / carrying cost].